Purpose: Do some data exploration prior to machine learning with XGBoost - specifically, what and how many samples have paired physiology & gene expression at the same treatment/time combination?<br>
Author: Anna Pardo<br>
Date initiated: Jan. 28, 2026

In [1]:
import pandas as pd
import numpy as np

In [2]:
# load TPM
tpm = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/TPM/Yg_toYgIS_allTPM_correctedmd_over1mil.txt",
                 sep="\t",header="infer")
tpm.head()

/tmp/ipykernel_512/576154366.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  tpm = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/TPM/Yg_toYgIS_allTPM_correctedmd_over1mil.txt",


,sample_name,genotype,time,treat,ZT,species,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g
0,Y1,18,1.0,W,1.0,gloriosa,34.002815,4.546167,0.0,17.236119,...,0.000000,6.167441,1.435253,0.279077,11.420139,0.662252,1.886709,6.675105,0.0,0.000000
1,Y10,2AB,1.5,W,3.0,gloriosa,40.070758,3.628454,0.0,14.918115,...,0.713535,2.197522,15.695904,1.193256,10.172780,0.000000,2.214483,12.125756,0.0,0.000000
2,Y100,2AB,6.5,W,23.0,gloriosa,47.402599,5.201760,0.0,17.406497,...,0.314746,2.261806,21.742515,1.798380,10.256681,1.748663,2.075757,25.215640,0.0,1.838780
3,Y101,2AB,1.0,D,1.0,gloriosa,57.062380,6.374324,0.0,10.567561,...,0.000000,1.072366,27.573939,1.164591,9.105769,1.257431,2.431447,4.508369,0.0,0.813682
4,Y103,2AB,1.0,D,1.0,gloriosa,34.679279,6.087451,0.0,11.115252,...,0.000000,0.914550,28.621412,0.869052,7.076224,0.515567,2.419222,4.625880,0.0,0.867418


In [16]:
# load Yg physiology data (from Karolina, merged with RNA metadata)
yglicor = pd.read_excel("/home/leviathan22/Yucca_genomics/rna_insilico_genome/yglor_rnalibs_physiology.xlsx")

In [17]:
yglicor.head()

,sample_name,collectionDate,genotype,time,treat,ZT,batch,RNAsample#,fullID,fullIDtreat,photo,cond
0,Y111,2016-04-01 00:00:00,18,1.0,D,1.0,2016.April,NaN,18.2.3.2015,18.2.3.2015_D_1,2.627596,0.026132
1,Y123,2016-04-01 00:00:00,18,1.0,D,1.0,2016.April,NaN,18.2.4.2015,18.2.4.2015_D_1,2.679172,0.021603
2,Y117,2016-04-01 00:00:00,18,1.0,D,NaN,2016.April,NaN,18.1.1.2015,18.1.1.2015_D_,NaN,NaN
3,Y120,2016-04-01 00:00:00,18,1.5,D,3.0,2016.April,NaN,18.1.1.2015,18.1.1.2015_D_3,1.211527,0.007856
4,Y125,2016-04-01 00:00:00,18,1.5,D,3.0,2016.April,NaN,18.2.3.2015,18.2.3.2015_D_3,1.289725,0.009783


In [18]:
yglicor = yglicor[["sample_name","genotype","treat","ZT","photo","cond"]]
yglicor.head()

,sample_name,genotype,treat,ZT,photo,cond
0,Y111,18,D,1.0,2.627596,0.026132
1,Y123,18,D,1.0,2.679172,0.021603
2,Y117,18,D,NaN,NaN,NaN
3,Y120,18,D,3.0,1.211527,0.007856
4,Y125,18,D,3.0,1.289725,0.009783


In [19]:
yglicor.dropna(axis=0,inplace=True)
yglicor = yglicor[["sample_name","genotype","treat","ZT","photo","cond"]]
yglicor.head()

/tmp/ipykernel_512/2297750243.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yglicor.dropna(axis=0,inplace=True)


,sample_name,genotype,treat,ZT,photo,cond
0,Y111,18,D,1.0,2.627596,0.026132
1,Y123,18,D,1.0,2.679172,0.021603
3,Y120,18,D,3.0,1.211527,0.007856
4,Y125,18,D,3.0,1.289725,0.009783
5,Y129,18,D,3.0,1.549238,0.010471


In [6]:
yglicor["ZT"].unique()

array([ 1.,  3.,  5.,  7.,  9., 11., 13., 15., 17., 19., 21., 23.])

In [7]:
tpm["ZT"].unique()

array([ 1.,  3., 23., 21.,  5.,  7.,  9., 13., 11., 15., 17., 19.])

In [8]:
yglicor["treat"].unique()

array(['D', 'W'], dtype=object)

In [9]:
tpm["treat"].unique()

array(['W', 'D'], dtype=object)

In [20]:
# merge yglicor with TPM data
## first: fix genotype columns in both dataframes (important)
yglicor["genotype"] = yglicor["genotype"].astype(str)
tpm["genotype"] = tpm["genotype"].astype(str)

In [21]:
yglicor["genotype"] = yglicor["genotype"].replace(to_replace="Eu",value="Eudy")

In [22]:
tpm["genotype"].unique()

array(['18', '2AB', '1AB', '19', '15', 'Eudy', 'G', '56', '36', '13',
       '45', '52', '43', '37', '48', '55', '70', '61', '51', '46', '53',
       '16', '6', '12', '20', '50'], dtype=object)

In [23]:
yglicor["genotype"].unique()

array(['18', '1AB', '2AB', '15', '46', 'Eudy', '13', '19', '45', 'G',
       '36', '37', '48', '53', '56', '16', '52', '51', '55', '61', '70',
       '43'], dtype=object)

In [24]:
# merge dataframes
mdf = yglicor.merge(tpm)

In [25]:
len(mdf["sample_name"].unique())

454

In [26]:
mdf.head()

,sample_name,genotype,treat,ZT,photo,cond,time,species,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,...,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g
0,Y111,18,D,1.0,2.627596,0.026132,1.0,gloriosa,48.380647,6.981615,...,0.000000,5.500249,2.171056,0.671994,8.838875,0.372084,2.805992,4.407968,0.0,0.000000
1,Y123,18,D,1.0,2.679172,0.021603,1.0,gloriosa,52.585871,5.331016,...,0.101045,11.410506,2.456701,0.464693,10.125277,0.327475,1.058391,4.115716,0.0,0.000000
2,Y120,18,D,3.0,1.211527,0.007856,1.5,gloriosa,41.192373,5.821853,...,0.000000,5.037032,2.061853,0.896509,10.437337,0.302887,1.945765,5.470556,0.0,0.212331
3,Y125,18,D,3.0,1.289725,0.009783,1.5,gloriosa,49.458436,6.239587,...,0.000000,6.796193,3.347733,0.445386,9.763749,0.422761,2.479682,5.295689,0.0,0.000000
4,Y129,18,D,3.0,1.549238,0.010471,1.5,gloriosa,45.312890,7.263022,...,0.116594,7.899789,1.574850,0.584944,9.166182,0.539808,1.944959,7.736828,0.0,0.000000


In [27]:
# drop time & species cols
mdf.drop(["time","species"],axis=1,inplace=True)

In [39]:
# save mdf
mdf.to_csv("./paired_TPM_physiology.txt",sep="\t",header=True,index=False)

In [29]:
# check reps for each genotype-treatment-ZT combo
reps = mdf.groupby(["genotype","treat","ZT"]).count()["sample_name"].reset_index()
reps.head()

,genotype,treat,ZT,sample_name
0,13,D,1.0,1
1,13,D,5.0,3
2,13,D,9.0,2
3,13,D,13.0,2
4,13,D,17.0,1


In [31]:
reps.sort_values(by=["sample_name"],ascending=False,inplace=True)
reps

,genotype,treat,ZT,sample_name
9,13,W,13.0,5
177,G,W,21.0,4
163,Eudy,W,13.0,4
105,36,D,5.0,4
106,36,D,9.0,4
...,...,...,...,...
147,56,D,21.0,1
46,19,D,5.0,1
151,56,W,13.0,1
155,Eudy,D,5.0,1


In [33]:
reps["genotype"].unique()

array(['13', 'G', 'Eudy', '36', '19', '56', '1AB', '2AB', '52', '45',
       '15', '18', '48', '43', '37'], dtype=object)

In [34]:
reps[reps["sample_name"]>=3]["genotype"].unique()

array(['13', 'G', 'Eudy', '36', '19', '56', '1AB', '2AB', '52', '45',
       '15', '18'], dtype=object)

In [38]:
reps[(reps["treat"]=="D")&(reps["sample_name"]>2)]

,genotype,treat,ZT,sample_name
105,36,D,5.0,4
106,36,D,9.0,4
107,36,D,13.0,4
108,36,D,17.0,4
109,36,D,21.0,4
45,19,D,1.0,4
170,G,D,17.0,4
91,2AB,D,23.0,3
90,2AB,D,21.0,3
1,13,D,5.0,3
